## Overview
This notebook processes machine learning predictions into gridded NetCDF files suitable for GPlates visualization by:
1. Loading point predictions from Workflow 2
2. Gridding data to regular lat/lon mesh
3. Applying spatial interpolation and smoothing
4. Performing temporal interpolation between timesteps
5. Converting between reference frames (Mantle ↔ Paleomagnetic)
6. Saving compressed NetCDF outputs

**Required time**: 2-6 hours depending on temporal range and parallel processing.

---

In [1]:
from pyDTDM import *
import warnings
import yaml
try:
    from yaml import Cloader as Loader
except ImportError:

    from yaml import Loader

import xarray as xr


/Users/ssin4735/miniforge3/envs/EBMTest311/lib/python3.11/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)
/Users/ssin4735/miniforge3/envs/EBMTest311/lib/python3.11/site-packages/ptt/documentation.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


Configuration and Setup

### Input Requirements:
- **Parquet Files**: Point predictions from Workflow 2
- **Plate Model**: Rotation files for reference frame conversions
- **Grid Parameters**: Resolution, compression settings

### Output Structure:
- `DL_Model_XX_MANTLE/`: Grids in mantle reference frame
- `DL_Model_Mantle_interpolated/`: Temporally smoothed grids
- `DL_Model_Paleomag/`: Grids in paleomagnetic reference frame


In [2]:
# Define the path to the configuration file
config_file = "InputFiles/Merdith1Ga.yaml"

# Open the configuration file and load parameters using YAML
with open(config_file) as f:
    PARAMS = yaml.load(f, Loader=Loader)  # Load parameters from YAML file using the specified Loader

# Print a confirmation message indicating the configuration file and parameters
print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")
print(" Parameters set from %s" % config_file)  # Display the path of the configuration file
print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")

––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– 
 Parameters set from InputFiles/Merdith1Ga.yaml
––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– 


In [3]:
# Input Files 
MODEL_NAME=PARAMS['InputFiles']['plate_kinematics']['model_name']
MODEL_DIR = PARAMS['InputFiles']['plate_kinematics']['model_dir']  ## plate model
topology_filenames =[f"{MODEL_DIR}/{i}" for i in PARAMS['InputFiles']['plate_kinematics']['topology_files']]
rotation_filenames = [f"{MODEL_DIR}/{i}" for i in PARAMS['InputFiles']['plate_kinematics']['rotation_files']]
agegrid=PARAMS['InputFiles']['plate_kinematics']['agegrid']

ETOPO_FILE=PARAMS['InputFiles']['Raster']['ETOPO_FILE'] # ETOPO grid in meters (can be netCDf or GeoTiff)
ETOPO_Type=PARAMS['InputFiles']['Raster']['Raster_type']
coastlines = f"{MODEL_DIR }/{PARAMS['InputFiles']['plate_kinematics']['coastline_file']}"
static_polygon_file=f"{MODEL_DIR }/{PARAMS['InputFiles']['plate_kinematics']['static_polygon']}"
static_polygons = pygplates.FeatureCollection(static_polygon_file)
continents=f"{MODEL_DIR }/{PARAMS['InputFiles']['plate_kinematics']['continents']}"
print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")
print("Reading input file..... \n")
print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")
print(f"Plate Model: {MODEL_NAME} \n")
print(f"Model Directory: {MODEL_DIR} \n")
print(f"Coastlines: {coastlines} \n")
print(f"Continents: {continents} \n")
print(f"Static Polygons: {static_polygon_file} \n")
print(f"Model Agegrid: {agegrid} \n")
print(f"ETopo grid: {ETOPO_FILE}")
print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– \n")

––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– 
Reading input file..... 

––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– 
Plate Model: Merdith1Ga 

Model Directory: /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions 

Coastlines: /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/StaticGeometries/GSHHS_l_coastlines.shp 

Continents: /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/shapes_continents.shp 

Static Polygons: /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/shapes_static_polygons.shp 

Model Agegrid: //Volumes/SatyamData/Merdith1Ga/Merdith_etal_OPTIMISED_gplately2/seafloor_grid_output/SEAFLOOR_AGE/masked/ 

ETopo grid: /Users/ssin4735/Documents/PROJECT/PhD Project/Codes and Data/Part 2/A_DEEPTIMETOPO/Data/Smoothed ETopo.tif
––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– 



In [4]:
Paleomag_ID=PARAMS['Parameters']['paleomag_id']
Mantle_ID=PARAMS['Parameters']['mantle_optimised_id']

#The initial positions of crustal points are evenly distributed within the designated region. 
# At mesh refinement level zero, the points are approximately 20 degrees apart.
# Each increase in the density level results in a halving of the spacing between points.
MESH_REFINEMENT_LEVEL=PARAMS['Parameters']['mesh_refinement_level']  # higher refinement level will take longer time to run for optimisation 
WINDOW_SIZE=int(PARAMS['Parameters']['time_window_size'])
Weighted=PARAMS['Parameters']['weighted_mean']


NETCDF_GRID_RESOLUTION=PARAMS['GridParameters']['grid_spacing']  # in degree
ZLIB=PARAMS['GridParameters']['compression']['zlib'] 
COMPLEVEL=PARAMS['GridParameters']['compression']['complevel'] 

FROM_TIME=int(PARAMS['TimeParameters']['time_max'])
TO_TIME=int(PARAMS['TimeParameters']['time_min'])
TIME_STEPS=int(PARAMS['TimeParameters']['time_step'])




parallel=PARAMS['Parameters']['number_of_cpus']### No of core to use or None for single core


print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")
print("The following parameters are set-")
print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")
print(f"Mantle Optmised Reference Frame ID: {Mantle_ID}")
print(f"Paleomagnetic Reference Frame ID: {Paleomag_ID} \n")

print(f"Moving Window Size: {WINDOW_SIZE}")
print(f"Weighted Mean: {Weighted}")

print(f"Mesh Refinement Level: {MESH_REFINEMENT_LEVEL}")
print(f"NetCDF GRID Resolution: {NETCDF_GRID_RESOLUTION}")
print(f"NetCDF Compression Level: {COMPLEVEL} \n")
print(f"Model Start Time: {FROM_TIME}")
print(f"Model End Time: {TO_TIME}")
print(f"Model Time Step: {TIME_STEPS}\n")


print(f"Number of CPU: {parallel}") # -1 means all the freely available CPU


print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")

––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– 
The following parameters are set-
––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– 
Mantle Optmised Reference Frame ID: 0
Paleomagnetic Reference Frame ID: 0 

Moving Window Size: 25
Weighted Mean: True
Mesh Refinement Level: 9
NetCDF GRID Resolution: 0.1
NetCDF Compression Level: 5 

Model Start Time: 200
Model End Time: 0
Model Time Step: 1

Number of CPU: -1
––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– 


In [ ]:
# Output Directory
OUTPUT_FOLDER=PARAMS['OutputFiles']['output_dir']

DEFAULT_OUTPUT_CSV=os.path.join(OUTPUT_FOLDER,'CSV')
DEFAULT_OUTPUT_NetCDF=os.path.join(OUTPUT_FOLDER,'NetCDF') # folder to store output NetCDF grid

ml_model_name="DL"



print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")
print(f"All the output will be saved in {OUTPUT_FOLDER}")
print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")
create_directory_if_not_exists(OUTPUT_FOLDER)
create_directory_if_not_exists(DEFAULT_OUTPUT_CSV)
create_directory_if_not_exists(DEFAULT_OUTPUT_NetCDF)
print("––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– ")


––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– 
All the output will be saved in /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/Paleotopography
––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– 
Directory already exists: /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/Paleotopography
Directory already exists: /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/Paleotopography/CSV
Directory already exists: /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/Paleotopography/NetCDF
––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––– 


## Define Plate Reconstruction

In [6]:
# rotation_model = pygplates.RotationModel(rotation_filenames)
# topology_features = pygplates.FeatureCollection()
# for topology_filename in topology_filenames:
#         topology_features.add( pygplates.FeatureCollection(topology_filename))


PK=PlateKinematicsParameters(topology_filenames, 
                             rotation_filenames,
                             static_polygons,
                             agegrid=agegrid,
                             coastlines=coastlines,
                             continents=continents,
                             anchor_plate_id=Mantle_ID)

time = 0 #Ma
gplot = gplately.PlotTopologies(PK.model, coastlines=coastlines, continents=continents, time=time)


In [7]:
all_times=glob.glob(f"{DEFAULT_OUTPUT_CSV}/Prediction_{WINDOW_SIZE}_NEW/*")
all_times=np.sort([int(time.split('_')[-1].split('.')[0].split('Ma')[0]) for time in all_times])

In [8]:
all_times

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  97,  98,  99, 100, 101, 102, 103, 104, 105,
       106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118,
       119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131,
       132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144,
       145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157,
       158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170,
       171, 172, 173, 174, 175, 176, 177, 179, 180, 181, 182, 18

## Spatial Gridding Process

### Step 1: Point-to-Grid Conversion
Converts scattered point predictions to regular grid:
- Uses `df_to_NetCDF()` with specified resolution (e.g., 0.1°)
- Bins data using `scipy.stats.binned_statistic_2d`
- Computes mean elevation within each grid cell
- Initial grid contains NaN (no-data) cells

### Step 2: Local Interpolation
Fills gaps near data points:
- `post_process_grid()`: KNN interpolation
- **Threshold Distance**: Maximum distance to search for neighbors (1°)
- **n_neighbors**: Number of nearby points to average (5-8)
- Only fills cells within threshold of actual data

### Step 3: Gaussian Smoothing
Applies spatial low-pass filter:
- `nan_gaussian_filter()`: Smooths while respecting NaN mask
- **Sigma**: Standard deviation of Gaussian kernel (4-5)
- **Radius**: Spatial extent of smoothing (2° typical)
- Removes high-frequency noise from gridding artifacts

### Step 4: Mask Preservation
**Critical**: Original NaN mask is reapplied after smoothing to prevent extrapolation into regions without data.

## Saving Processed grid as NetCDF

In [ ]:
def process_and_save_netcdf(reconstruction_time):
    try:
        old = np.seterr(divide='ignore', invalid='ignore')
        print(f"Working on Time={reconstruction_time} Ma")

        # -----------------------------
        # Load input parquet data
        # -----------------------------
        # data_path = f'{DEFAULT_OUTPUT_CSV}/Prediction/Predicted_{MODEL_NAME}_{reconstruction_time}Ma.parquet'
        data_path = f'{DEFAULT_OUTPUT_CSV}/Prediction_{WINDOW_SIZE}_NEW/Predicted_{MODEL_NAME}_{reconstruction_time}Ma.parquet'
        Data = pd.read_parquet(data_path)
        # print(len(Data))
        Data = Data[(Data['Trench Distance'] < 1000000) & (Data['Trench Distance'] > 100000)]
        print(len(Data))
        column_for_netcdf1 = f"Elevation{ml_model_name}"

        compression = {'zlib': ZLIB, 'complevel': COMPLEVEL}
        encoding = {column_for_netcdf1: compression} if compression else None

        # -----------------------------
        # Create grid
        # -----------------------------
        da = df_to_NetCDF(
            x=Data['Longitude'],
            y=Data['Latitude'],
            z=Data[column_for_netcdf1],
            statistic='mean',
            grid_resolution=NETCDF_GRID_RESOLUTION,
            clip=(None, None),
            lat_bin_edges=np.arange(-90, 90 + NETCDF_GRID_RESOLUTION, NETCDF_GRID_RESOLUTION),
            lon_bin_edges=np.arange(-180, 180 + NETCDF_GRID_RESOLUTION, NETCDF_GRID_RESOLUTION),
        )

        db = da.to_dataset(name=column_for_netcdf1)

        # Keep original NaN mask (before interpolation)
        # original_mask = np.isnan(db[column_for_netcdf1].values)

        # -----------------------------
        # Step 1: Local interpolation
        # -----------------------------
        da_interp = post_process_grid(
            db[column_for_netcdf1], 
            threshold_distance=10, 
            n_neighbors=5
        )
            # Keep original NaN mask (before interpolation)
        original_mask = np.isnan(da_interp)
        # da_interp =gplately.grids.fill_raster(da.values)
        # -----------------------------
        # Step 2: Gaussian smoothing
        # -----------------------------
        smoothed = nan_gaussian_filter(da_interp, sigma=4, radius=2)
        # smoothed = da_interp
        # -----------------------------
        # Step 3: Reapply NaN mask
        # -----------------------------
        smoothed[original_mask] = np.nan

        ds_smooth = xr.Dataset(
            {column_for_netcdf1: (('Latitude', 'Longitude'), smoothed)},
            coords={'Latitude': db['Latitude'].values, 'Longitude': db['Longitude'].values},
        )
    #     ds_smooth = xr.Dataset(
    #     {column_for_netcdf1: (('Latitude', 'Longitude'), da_interp.values)},
    #     coords={'Latitude': db['Latitude'].values, 'Longitude': db['Longitude'].values},
    # )


        # -----------------------------
        # Save to NetCDF
        # -----------------------------
        output_dir = f"{DEFAULT_OUTPUT_NetCDF}/{ml_model_name}_Model_{WINDOW_SIZE}_MANTLE"
        create_directory_if_not_exists(output_dir)

        output_file = f"{output_dir}/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc"

        if os.path.exists(output_file):
            os.remove(output_file)
            print("File deleted successfully")
        else:
            print("File does not exist")

        ds_smooth.attrs['title'] = 'Paleotopography Predictions'
        ds_smooth.attrs['source'] = f'{ml_model_name} Model'
        ds_smooth.attrs['model_name'] = f'{ml_model_name}_Model'
        ds_smooth.attrs['plate_model'] = f'{MODEL_NAME}'
        ds_smooth.attrs['reference_frame'] = 'Mantle-optimized'
        ds_smooth.attrs['window_size'] = f'{WINDOW_SIZE} Ma'
        ds_smooth.attrs['grid_resolution'] = f'{NETCDF_GRID_RESOLUTION} degrees'
        ds_smooth.attrs['creation_date'] = pd.Timestamp.now().isoformat()

        ds_smooth.to_netcdf(output_file, encoding={column_for_netcdf1: compression})
        print(f"Saved NetCDF: {output_file}")

        return ds_smooth, original_mask,smoothed
    except Exception as e:
        print(f"Error processing time {reconstruction_time} Ma: {e}")
        return None, None,None


In [40]:
ds_smooth=process_and_save_netcdf(0)

Working on Time=0 Ma
222728
210609
Interpolated 356470 points
Created directory: /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/Paleotopography/NetCDF/DL_Model_25_MANTLE
File does not exist
Saved NetCDF: /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/Paleotopography/NetCDF/DL_Model_25_MANTLE/DL_Model_Merdith1Ga_0.nc


In [ ]:
# Use joblib to parallelize the loop
Parallel(n_jobs=6)(delayed(process_and_save_netcdf)(t) for t in all_times)


## ⏱Temporal Filling/Interpolation


#### 1. Load Time Series
- Reads NetCDF grids for consecutive timesteps
- Combines into single xarray Dataset with time dimension
- Sorts by time (oldest to youngest)

#### 2. Temporal Gap Filling
```python
combined_ds_filled = combined_ds.interpolate_na(dim='time', method='linear', max_gap=6)
```
- Linear interpolation along time dimension
- **max_gap**: Maximum number of missing timesteps to bridge (6 Ma)
- Prevents unrealistic interpolation across large gaps

#### 3. Resample to 1 Ma Intervals
```python
interpolated_ds = combined_ds_filled.interp(time=target_times)
```
- Creates grids at every 1 Ma between existing timesteps
- Uses linear interpolation for elevation values

#### 4. Temporal Smoothing
```python
window_size = 5  # Ma
smoothed = data.rolling(time=window_size, center=False).mean()
```
- **Moving Average**: Averages elevation over 5 Ma window
- Reduces temporal noise from independent model predictions
- Makes time series visually smoother


In [ ]:

ml_model_name = "DL"  # or "RFC" or "RF"
# create_directory_if_not_exists(f"{DEFAULT_OUTPUT_NetCDF}/{WINDOW_SIZE}_processed_interpolated/{ml_model_name}_Model")
output_dir = f"{DEFAULT_OUTPUT_NetCDF}/{ml_model_name}_Model_{WINDOW_SIZE}_Mantle_interpolated"
create_directory_if_not_exists(output_dir)

def moving_average(data, window_size):
    return data.rolling(time=window_size, center=False).mean()

# Load all your NetCDF files into a list of DataArrays
# reconstruction_times = all_times[:138] # Your list of reconstruction times

# all_times = np.arange(360, 520, 1)  # Example: replace with your actual times

_window_start=[200,120,40,0]

for i in  range(len(_window_start)-1):

    all_times=np.arange(_window_start[i+1],_window_start[i]+4, 1)
    data_arrays = []

    print("Loading NetCDF files...")
    times=[]
    for reconstruction_time in all_times:
        try:
            # file_path =f"{DEFAULT_OUTPUT_NetCDF}/{ml_model_name}_Model_MANTLE/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc"
            file_path=f"{DEFAULT_OUTPUT_NetCDF}/{ml_model_name}_Model_{WINDOW_SIZE}_MANTLE/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc"
            ds = xr.open_dataset(file_path)
            data_arrays.append(ds)
            print(f"Loaded {file_path}")
            times.append(reconstruction_time)
        except FileNotFoundError:
            print(f"File not found: {file_path}. Skipping this time step.")
            pass


    # Combine them into a single Dataset with 'time' as a new dimension
    print("Combining datasets...")
    combined_ds = xr.concat(data_arrays, dim='time')
    combined_ds['time'] = times

    # Sort the dataset by time (in case it's not already sorted)
    combined_ds = combined_ds.sortby('time')

    # Fill NaN values by linear interpolation over time
    print("Filling NaN values...")
    combined_ds_filled = combined_ds.interpolate_na(dim='time', method='linear', max_gap=6)

    # Create the target time array with 1 Ma intervals
    print("Creating target time array...")
    target_times = np.arange(min(all_times), max(all_times) + 1, 1)

    # Perform the interpolation to every 1 Ma interval
    print("Interpolating to every 1 Ma interval...")
    interpolated_ds = combined_ds_filled.interp(time=target_times)

    # Apply a moving average for temporal smoothing
    print("Applying temporal smoothing...")
    window_size = 5  # Adjust this based on your needs (e.g., smoothing over 5 Ma)
    smoothed_ds = interpolated_ds.copy()

    for var in smoothed_ds.data_vars:
        smoothed_ds[var] = moving_average(smoothed_ds[var], window_size=window_size)
        print(f"Applied smoothing to {var}")

    # Define zlib compression options for each variable
    compression_settings = {var: {'zlib': True, 'complevel': 9} for var in smoothed_ds.data_vars}

    # Save the smoothed datasets at each 1 Ma interval with zlib compression
    print("Saving smoothed datasets...")

    # Save the smoothed datasets at each 1 Ma interval with zlib compression
    print("Saving smoothed datasets...")

    for reconstruction_time in target_times:
        smoothed_file_path = f"{output_dir}/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc"

        if os.path.exists(smoothed_file_path):
            os.remove(smoothed_file_path)
            print("File deleted successfully")
        else:
            print("File does not exist")
        smoothed_ds.sel(time=reconstruction_time).to_netcdf(smoothed_file_path, encoding=compression_settings)
        print(f"Saved {smoothed_file_path}")

    print("Process completed.")


Created directory: /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/Paleotopography/NetCDF/DL_Model_25_Mantle_interpolated
Loading NetCDF files...
Loaded /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/Paleotopography/NetCDF/DL_Model_25_MANTLE/DL_Model_Merdith1Ga_120.nc
Loaded /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/Paleotopography/NetCDF/DL_Model_25_MANTLE/DL_Model_Merdith1Ga_121.nc
Loaded /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/Paleotopography/NetCDF/DL_Model_25_MANTLE/DL_Model_Merdith1Ga_122.nc
Loaded /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/Paleotopography/NetCDF/DL_Model_25_MANTLE/DL_Model_Merdith1Ga_123.nc
Loaded /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/Paleotopography/NetCDF/DL_Model_25_MANTLE/DL_Model_Merdith1Ga_124.nc
Loaded /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/Paleotopography/NetCDF/DL_Model_25_MANTLE/DL_Model_Merdith1Ga_125.nc
Loaded /Volumes/SatyamData/Merdith1Ga/1Ga_Reconstructions/Paleotopography/NetCDF/DL_Model

In [11]:
import numpy as np
import xarray as xr

spatial_resolution = 0.1
temporal_resolution = 5
end_time = 100
data_vars = {}

for reconstruction_time in range(0, end_time, int(temporal_resolution)):
    try:
        print(f"Working on {reconstruction_time}")

        # Predicted dataset
        db = xr.open_dataset(
            f"{DEFAULT_OUTPUT_NetCDF}/{ml_model_name}_Model_Mantle_interpolated/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc"
        )

        db = db.coarsen(
            Latitude=int(10 * spatial_resolution),
            Longitude=int(10 * spatial_resolution)
        ).mean()

        # Extract the main variable
        elev = db["ElevationDL"]

        # Store as DataArray
        data_vars[f"Reconstruction_Time_{reconstruction_time}"] = (
            ("Latitude", "Longitude"), elev.values
        )

    except FileNotFoundError:
        print(f"Skipping time={reconstruction_time}, file not found")
    except Exception as e:
        print(f"Error at time={reconstruction_time}: {e}")

# Build Dataset
paleoelevation_ds = xr.Dataset(
    data_vars=data_vars,
    coords={
        "Latitude": elev["Latitude"].values,
        "Longitude": elev["Longitude"].values,
    }
)

# Save
paleoelevation_ds.to_netcdf(
    f"{DEFAULT_OUTPUT_NetCDF}/{ml_model_name}_Paleoelevation_Grids.nc"
)
print("All grids saved as multiple variables in a single NetCDF")


Working on 0
Working on 5
Working on 10
Working on 15
Working on 20
Working on 25
Working on 30
Working on 35
Working on 40
Working on 45
Working on 50
Working on 55
Working on 60
Working on 65
Working on 70
Working on 75
Working on 80
Working on 85
Working on 90
Working on 95
All grids saved as multiple variables in a single NetCDF


In [20]:
del paleoelevation_ds


## 🔄 Reference Frame Conversion

### Why Convert Reference Frames?

**Mantle Reference Frame**: 
- Used for dynamic topography modeling
- Plate motions relative to deep mantle (convection)
- Minimizes net lithospheric rotation

**Paleomagnetic Reference Frame**:
- Used for climate modeling and paleogeography
- Constrained by apparent polar wander paths
- Positions continents relative to Earth's rotation axis



#### Technical Details:
1. **Grid Reconstruction**: Assigns plate IDs to each grid cell
2. **Rotation**: Applies finite rotation between reference frames
3. **Regridding**: Interpolates rotated data back to regular grid
4. **Masking**: Preserves land/ocean boundaries

**Use GPlates' `Raster.rotate_reference_frames()`** built-in function for proper handling of plate boundaries.



## Change From Mantle Reference Frame to Paleomag Reference Frame

In [ ]:


climate_rotation_filenames = [f"{MODEL_DIR}/{i}" for i in PARAMS['InputFiles']['climate_parameters']['rotation_files']]


CM=ClimateParameters(topology_filenames, 
                             climate_rotation_filenames,
                             static_polygons,
                             agegrid=agegrid,
                             coastlines=coastlines,
                             continents=continents,
                             anchor_plate_id=Paleomag_ID)



In [ ]:
from gplately import Raster
create_directory_if_not_exists(f"{DEFAULT_OUTPUT_NetCDF}/{ml_model_name}_Model_Paleomag")

for reconstruction_time in all_times:
    print(f"Rotating grids:{reconstruction_time} Ma")
    da=xr.open_dataset(f"{DEFAULT_OUTPUT_NetCDF}/{ml_model_name}_Model_MANTLE/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc")
    raster=Raster(data=da.ElevationDL, plate_reconstruction=PK.model, extent='global',  time=reconstruction_time)
    
    raster.rotate_reference_frames(grid_spacing_degrees=NETCDF_GRID_RESOLUTION,
                                   reconstruction_time=reconstruction_time, 
                                   from_rotation_features_or_model=PK.rotation_model, 
                                   to_rotation_features_or_model=CM.rotation_model, 
                                   from_rotation_reference_plate=Mantle_ID, 
                                   to_rotation_reference_plate=Paleomag_ID, 
                                   non_reference_plate=701, 
                                   output_name= f"{DEFAULT_OUTPUT_NetCDF}/{ml_model_name}_Model_Paleomag/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc")



In [20]:
from joblib import Parallel, delayed
import xarray as xr
from gplately import Raster
import os

# Function to process a single reconstruction time
def rotate_grid(reconstruction_time):
    print(f"Rotating grids: {reconstruction_time} Ma")
    
    # Open the NetCDF file
    da = xr.open_dataset(f"{DEFAULT_OUTPUT_NetCDF}/{ml_model_name}_Model_MANTLE/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc")
    
    # Create Raster object
    raster = Raster(
        data=da.ElevationDL,
        plate_reconstruction=PK.model,
        extent='global',
        time=reconstruction_time
    )
    
    # Rotate the reference frame
    raster.rotate_reference_frames(
        grid_spacing_degrees=NETCDF_GRID_RESOLUTION,
        reconstruction_time=reconstruction_time,
        from_rotation_features_or_model=PK.rotation_model,
        to_rotation_features_or_model=CM.rotation_model,
        from_rotation_reference_plate=Mantle_ID,
        to_rotation_reference_plate=Paleomag_ID,
        non_reference_plate=701,
        output_name=f"{DEFAULT_OUTPUT_NetCDF}/{ml_model_name}_Model_Paleomag/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc"
    )

# Ensure output directory exists
os.makedirs(f"{DEFAULT_OUTPUT_NetCDF}/{ml_model_name}_Model_Paleomag", exist_ok=True)

# Run in parallel
Parallel(n_jobs=-1)(delayed(rotate_grid)(time) for time in all_times)


/Users/ssin4735/miniforge3/envs/EBMTest311/lib/python3.11/site-packages/ptt/documentation.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


Rotating grids: 0 Ma


/Users/ssin4735/miniforge3/envs/EBMTest311/lib/python3.11/site-packages/ptt/documentation.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
/Users/ssin4735/miniforge3/envs/EBMTest311/lib/python3.11/site-packages/ptt/documentation.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


Rotating grids: 1 Ma


/Users/ssin4735/miniforge3/envs/EBMTest311/lib/python3.11/site-packages/ptt/documentation.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


...

/Users/ssin4735/miniforge3/envs/EBMTest311/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.


Rotating grids: 173 Ma
Rotating grids: 174 Ma
Rotating grids: 175 Ma


/Users/ssin4735/miniforge3/envs/EBMTest311/lib/python3.11/site-packages/ptt/documentation.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


Rotating grids: 176 Ma
Rotating grids: 177 Ma
Rotating grids: 178 Ma
Rotating grids: 179 Ma
Rotating grids: 180 Ma
Rotating grids: 181 Ma
Rotating grids: 182 Ma
Rotating grids: 183 Ma
Rotating grids: 184 Ma
Rotating grids: 185 Ma
Rotating grids: 186 Ma
Rotating grids: 187 Ma
Rotating grids: 188 Ma
Rotating grids: 189 Ma
Rotating grids: 190 Ma
Rotating grids: 191 Ma
Rotating grids: 192 Ma
Rotating grids: 193 Ma
Rotating grids: 194 Ma
Rotating grids: 197 Ma
Rotating grids: 198 Ma
Rotating grids: 199 Ma
Rotating grids: 200 Ma
Rotating grids: 201 Ma
Rotating grids: 202 Ma
Rotating grids: 203 Ma
Rotating grids: 204 Ma
Rotating grids: 205 Ma
Rotating grids: 206 Ma
Rotating grids: 207 Ma
Rotating grids: 208 Ma
Rotating grids: 209 Ma
Rotating grids: 210 Ma
Rotating grids: 211 Ma
Rotating grids: 212 Ma
Rotating grids: 213 Ma
Rotating grids: 214 Ma
Rotating grids: 215 Ma
Rotating grids: 216 Ma
Rotating grids: 217 Ma
Rotating grids: 218 Ma
Rotating grids: 219 Ma
Rotating grids: 220 Ma
Rotating gr

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [ ]:
import xarray as xr
import numpy as np


ml_model_name = "DL"  # or "RFC" or "RF"
# create_directory_if_not_exists(f"{DEFAULT_OUTPUT_NetCDF}/{WINDOW_SIZE}_processed_interpolated/{ml_model_name}_Model")
output_dir = f"{DEFAULT_OUTPUT_NetCDF}/{ml_model_name}_Model_Mantle_interpolated"
create_directory_if_not_exists(output_dir)

def moving_average(data, window_size):
    return data.rolling(time=window_size, center=False).mean()

# Load all your NetCDF files into a list of DataArrays
# reconstruction_times = all_times[:138] # Your list of reconstruction times

all_times = np.arange(440, 470, 1)  # Example: replace with your actual times

data_arrays = []

print("Loading NetCDF files...")
times=[]
for reconstruction_time in all_times:
    try:
        file_path =f"{DEFAULT_OUTPUT_NetCDF}/{ml_model_name}_Model_MANTLE/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc"
        ds = xr.open_dataset(file_path)
        data_arrays.append(ds)
        print(f"Loaded {file_path}")
        times.append(reconstruction_time)
    except FileNotFoundError:
        print(f"File not found: {file_path}. Skipping this time step.")
        pass


# Combine them into a single Dataset with 'time' as a new dimension
print("Combining datasets...")
combined_ds = xr.concat(data_arrays, dim='time')
combined_ds['time'] = times

# Sort the dataset by time (in case it's not already sorted)
combined_ds = combined_ds.sortby('time')

# Fill NaN values by linear interpolation over time
print("Filling NaN values...")
combined_ds_filled = combined_ds.interpolate_na(dim='time', method='linear', max_gap=6)

# Create the target time array with 1 Ma intervals
print("Creating target time array...")
target_times = np.arange(min(all_times), max(all_times) + 1, 1)

# Perform the interpolation to every 1 Ma interval
print("Interpolating to every 1 Ma interval...")
interpolated_ds = combined_ds_filled.interp(time=target_times)

# Apply a moving average for temporal smoothing
print("Applying temporal smoothing...")
window_size = 5  # Adjust this based on your needs (e.g., smoothing over 5 Ma)
smoothed_ds = interpolated_ds.copy()

for var in smoothed_ds.data_vars:
    smoothed_ds[var] = moving_average(smoothed_ds[var], window_size=window_size)
    print(f"Applied smoothing to {var}")

# Define zlib compression options for each variable
compression_settings = {var: {'zlib': True, 'complevel': 9} for var in smoothed_ds.data_vars}

# Save the smoothed datasets at each 1 Ma interval with zlib compression
print("Saving smoothed datasets...")

# Save the smoothed datasets at each 1 Ma interval with zlib compression
print("Saving smoothed datasets...")

for reconstruction_time in target_times:
    smoothed_file_path = f"{output_dir}/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc"
    smoothed_ds.sel(time=reconstruction_time).to_netcdf(smoothed_file_path, encoding=compression_settings)
    print(f"Saved {smoothed_file_path}")

print("Process completed.")


Loading NetCDF files...
Loaded /Volumes/SatyamData/Paper3/NETCDF/DL_ModelFinal_processed/DL_Model_phase2_440.nc
Loaded /Volumes/SatyamData/Paper3/NETCDF/DL_ModelFinal_processed/DL_Model_phase2_441.nc
Loaded /Volumes/SatyamData/Paper3/NETCDF/DL_ModelFinal_processed/DL_Model_phase2_442.nc
Loaded /Volumes/SatyamData/Paper3/NETCDF/DL_ModelFinal_processed/DL_Model_phase2_443.nc
Loaded /Volumes/SatyamData/Paper3/NETCDF/DL_ModelFinal_processed/DL_Model_phase2_444.nc
Loaded /Volumes/SatyamData/Paper3/NETCDF/DL_ModelFinal_processed/DL_Model_phase2_445.nc
Loaded /Volumes/SatyamData/Paper3/NETCDF/DL_ModelFinal_processed/DL_Model_phase2_446.nc
Loaded /Volumes/SatyamData/Paper3/NETCDF/DL_ModelFinal_processed/DL_Model_phase2_447.nc
Loaded /Volumes/SatyamData/Paper3/NETCDF/DL_ModelFinal_processed/DL_Model_phase2_448.nc
Loaded /Volumes/SatyamData/Paper3/NETCDF/DL_ModelFinal_processed/DL_Model_phase2_449.nc
Loaded /Volumes/SatyamData/Paper3/NETCDF/DL_ModelFinal_processed/DL_Model_phase2_450.nc
Loaded /

In [ ]:
# Save the smoothed datasets at each 1 Ma interval with zlib compression
print("Saving smoothed datasets...")

for reconstruction_time in target_times:
    smoothed_file_path = f'{DEFAULT_OUTPUT_NetCDF}/{WINDOW_SIZE}_processed_interpolated/{ml_model_name}_Model/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc'
    smoothed_ds.sel(time=reconstruction_time).to_netcdf(smoothed_file_path, encoding=compression_settings)
    print(f"Saved {smoothed_file_path}")

print("Process completed.")


In [35]:
from gplately import Raster
create_directory_if_not_exists(f"/Volumes/SatyamData/Paper4_HirnatianGlaciation/PaleomagElevation")

for reconstruction_time in all_times[:400]:
    print(f"Rotating grids:{reconstruction_time} Ma")
    da=xr.open_dataset(f"/Volumes/SatyamData/Paper3/NetCDFs1000km_Processed/DL_Model_Phase2D_{reconstruction_time}.nc")
    raster=Raster(data=da.ElevationDL, plate_reconstruction=PK.model, extent='global',  time=reconstruction_time)
    
    raster.rotate_reference_frames(grid_spacing_degrees=NETCDF_GRID_RESOLUTION,
                                   reconstruction_time=reconstruction_time, 
                                   from_rotation_features_or_model=PK.rotation_model, 
                                   to_rotation_features_or_model=PK.rotation_model, 
                                   from_rotation_reference_plate=Mantle_ID, 
                                   to_rotation_reference_plate=Paleomag_ID, 
                                   non_reference_plate=701, 
                                   output_name= f"/Volumes/SatyamData/Paper4_HirnatianGlaciation/PaleomagElevation/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc")



Rotating grids:0 Ma
Rotating grids:1 Ma
Rotating grids:2 Ma
Rotating grids:3 Ma
Rotating grids:4 Ma
Rotating grids:5 Ma
Rotating grids:6 Ma
Rotating grids:7 Ma
Rotating grids:8 Ma
Rotating grids:9 Ma
Rotating grids:10 Ma
Rotating grids:11 Ma
Rotating grids:12 Ma
Rotating grids:13 Ma
Rotating grids:14 Ma
Rotating grids:15 Ma
Rotating grids:16 Ma
Rotating grids:17 Ma
Rotating grids:18 Ma
Rotating grids:19 Ma
Rotating grids:20 Ma
Rotating grids:21 Ma
Rotating grids:22 Ma
Rotating grids:23 Ma
Rotating grids:24 Ma
Rotating grids:25 Ma
Rotating grids:26 Ma
Rotating grids:27 Ma
Rotating grids:28 Ma
Rotating grids:29 Ma
Rotating grids:30 Ma
Rotating grids:31 Ma
Rotating grids:32 Ma
Rotating grids:33 Ma
Rotating grids:34 Ma
Rotating grids:35 Ma
Rotating grids:36 Ma
Rotating grids:37 Ma
Rotating grids:38 Ma
Rotating grids:39 Ma
Rotating grids:40 Ma
Rotating grids:41 Ma
Rotating grids:42 Ma
Rotating grids:43 Ma
Rotating grids:44 Ma
Rotating grids:45 Ma
Rotating grids:46 Ma
Rotating grids:47 Ma
Ro

In [8]:
ml_model_name="DL"

In [17]:
import xarray as xr
import numpy as np
import os

def create_directory_if_not_exists(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def moving_average_over_time(ds_list, window_size):
    stacked = xr.concat(ds_list, dim='time')
    # smoothed = stacked.rolling(time=window_size, center=False).mean().isel(time=-1)
    smoothed = stacked.rolling(time=window_size, center=False).mean()
    return smoothed.isel(time=0) 

# Setup
output_dir = f"/Volumes/SatyamData/Paper3/NETCDF/Mantle/{ml_model_name}_Model_interpolated"
create_directory_if_not_exists(output_dir)
print("Starting streaming interpolation and smoothing...")

all_times = np.arange(0, 526, 1)  # Example times from 0 to 525 Ma
target_times = np.arange(0, 526, 1)
window_size = 5
buffer = []  # to store the window for moving average

# Loop over target times
for i, t in enumerate(target_times):
    print(f"Processing target time {t} Ma")

    # Collect 1 dataset or interpolate from neighbors
    before_time = max(all_times[0], t - 1)
    after_time = min(all_times[-1], t + 1)

    # before_time = max(all_times[0], t - 3)
    # after_time = t  # t is the later time now

    # Get nearest actual files for interpolation
    try:
        ds_before = xr.open_dataset(f"/Volumes/SatyamData/Paper3/NETCDF/DL_ModelFinal_processed/{ml_model_name}_Model_{MODEL_NAME}_{before_time}.nc")
        ds_after = xr.open_dataset(f"/Volumes/SatyamData/Paper3/NETCDF/DL_ModelFinal_processed/{ml_model_name}_Model_{MODEL_NAME}_{after_time}.nc")

        ds_before = ds_before.expand_dims("time")
        ds_before["time"] = [before_time]
        ds_after = ds_after.expand_dims("time")
        ds_after["time"] = [after_time]

        interpolated = xr.concat([ds_before, ds_after], dim="time").interp(time=[t])
        interpolated = interpolated.isel(time=0)  # single time slice
        buffer.append(interpolated)

        # Maintain rolling window
        if len(buffer) > window_size:
            buffer.pop(0)

        if len(buffer) == window_size:
            smoothed = moving_average_over_time(buffer, window_size)

            smoothed_file_path = f"{output_dir}/{ml_model_name}_Model_{MODEL_NAME}_{t}.nc"
            smoothed.expand_dims("time").to_netcdf(smoothed_file_path, encoding={var: {'zlib': True, 'complevel': 9} for var in smoothed.data_vars})
            print(f"Saved {smoothed_file_path}")
        else:
            print(f"Skipping time {t} (insufficient data for smoothing window)")

    except FileNotFoundError as e:
        print(f"Missing file for {t}: {e}")
        continue

print("Completed streaming interpolation and smoothing.")


Starting streaming interpolation and smoothing...
Processing target time 0 Ma
Skipping time 0 (insufficient data for smoothing window)
Processing target time 1 Ma
Skipping time 1 (insufficient data for smoothing window)
Processing target time 2 Ma
Skipping time 2 (insufficient data for smoothing window)
Processing target time 3 Ma
Skipping time 3 (insufficient data for smoothing window)
Processing target time 4 Ma
Saved /Volumes/SatyamData/Paper3/NETCDF/Mantle/DL_Model_interpolated/DL_Model_phase2_4.nc
Processing target time 5 Ma
Saved /Volumes/SatyamData/Paper3/NETCDF/Mantle/DL_Model_interpolated/DL_Model_phase2_5.nc
Processing target time 6 Ma
Saved /Volumes/SatyamData/Paper3/NETCDF/Mantle/DL_Model_interpolated/DL_Model_phase2_6.nc
Processing target time 7 Ma
Saved /Volumes/SatyamData/Paper3/NETCDF/Mantle/DL_Model_interpolated/DL_Model_phase2_7.nc
Processing target time 8 Ma
Saved /Volumes/SatyamData/Paper3/NETCDF/Mantle/DL_Model_interpolated/DL_Model_phase2_8.nc
Processing target ti

In [11]:
from gplately import Raster



ml_model_name="DL"
MODEL_NAME='phase2'
# Setup
# output_dir = f"/Volumes/SatyamData/Paper3/NETCDF/Mantle/{ml_model_name}_Model_interpolated"
output_dir = f"/Volumes/SatyamData/Paper3/NETCDF/Paleomag/{ml_model_name}_Model_interpolated"
create_directory_if_not_exists(output_dir)
# print("Starting streaming interpolation and smoothing...")

all_times = np.arange(0, 526, 1)  # Example times from 0 to 525 Ma
# target_times = np.arange(0, 526, 1)




# create_directory_if_not_exists(f"/Volumes/SatyamData/Paper4_HirnatianGlaciation/PaleomagElevation")

for reconstruction_time in all_times:
    print(f"Rotating grids:{reconstruction_time} Ma")
    da=xr.open_dataset(f"/Volumes/SatyamData/Paper3/NETCDF/Mantle/PaleoDM/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc")
    raster=Raster(data=da.ElevationDL, plate_reconstruction=PK.model, extent='global',  time=reconstruction_time)
    
    raster.rotate_reference_frames(grid_spacing_degrees=NETCDF_GRID_RESOLUTION,
                                   reconstruction_time=reconstruction_time, 
                                   from_rotation_features_or_model=PK.rotation_model, 
                                   to_rotation_features_or_model=PK.rotation_model, 
                                   from_rotation_reference_plate=Mantle_ID, 
                                   to_rotation_reference_plate=Paleomag_ID, 
                                   non_reference_plate=701, 
                                   output_name= f"/Volumes/SatyamData/Paper3/NETCDF/Paleomag/{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc")



Rotating grids:0 Ma
Rotating grids:1 Ma
Rotating grids:2 Ma
Rotating grids:3 Ma
Rotating grids:4 Ma
Rotating grids:5 Ma
Rotating grids:6 Ma
Rotating grids:7 Ma
Rotating grids:8 Ma
Rotating grids:9 Ma
Rotating grids:10 Ma
Rotating grids:11 Ma
Rotating grids:12 Ma
Rotating grids:13 Ma
Rotating grids:14 Ma
Rotating grids:15 Ma
Rotating grids:16 Ma
Rotating grids:17 Ma
Rotating grids:18 Ma
Rotating grids:19 Ma
Rotating grids:20 Ma
Rotating grids:21 Ma
Rotating grids:22 Ma
Rotating grids:23 Ma
Rotating grids:24 Ma
Rotating grids:25 Ma
Rotating grids:26 Ma
Rotating grids:27 Ma
Rotating grids:28 Ma
Rotating grids:29 Ma
Rotating grids:30 Ma
Rotating grids:31 Ma
Rotating grids:32 Ma
Rotating grids:33 Ma
Rotating grids:34 Ma
Rotating grids:35 Ma
Rotating grids:36 Ma
Rotating grids:37 Ma
Rotating grids:38 Ma
Rotating grids:39 Ma
Rotating grids:40 Ma
Rotating grids:41 Ma
Rotating grids:42 Ma
Rotating grids:43 Ma
Rotating grids:44 Ma
Rotating grids:45 Ma
Rotating grids:46 Ma
Rotating grids:47 Ma
Ro

---

## NetCDF Compression

### Why Compress?
- Raw grids: ~50-100 MB per timestep
- Compressed: ~5-10 MB per timestep
- Essential for long time series (500+ timesteps)

### Settings:
```python
compression = {'zlib': True, 'complevel': 9}
encoding = {'ElevationDL': compression}
ds.to_netcdf(output_file, encoding=encoding)
```

- **zlib**: Lossless compression algorithm
- **complevel**: Compression level (1-9, higher = smaller but slower)
- **9**: Maximum compression (recommended for archival)

---

## 🎨 Multi-Variable NetCDF Output

### Structure:
Some workflows create single NetCDF with multiple time slices as separate variables:

```python
paleoelevation_ds = xr.Dataset(
    data_vars={
        "Reconstruction_Time_0": (("Lat", "Lon"), data_0),
        "Reconstruction_Time_20": (("Lat", "Lon"), data_20),
        ...
    }
)
```

**Pros**: 
- Single file easier to manage
- Faster loading for visualization

**Cons**:
- Large file size (even compressed)
- Difficult to update single timestep


---

## 🔧 Parallel Processing

### Spatial Gridding:
```python
Parallel(n_jobs=6)(delayed(process_and_save_netcdf)(t) for t in all_times)
```
- Each timestep processed independently
- Uses `joblib` for parallelization
- **n_jobs**: Number of parallel workers (6 recommended)

### Reference Frame Conversion:
```python
Parallel(n_jobs=-1)(delayed(rotate_grid)(time) for time in all_times)
```
- **n_jobs=-1**: Use all available CPU cores
- Rotation is computationally expensive
- Good candidate for parallelization

---

## 🎯 Quality Control

### Spatial Checks:
- **Visual Inspection**: Plot random timesteps to check for artifacts
- **NaN Coverage**: Ensure NaN regions make sense (oceans, far-field)
- **Smoothness**: No unrealistic discontinuities
- **Range**: Elevations within plausible bounds (-8000 to +8000 m)

### Temporal Checks:
- **Continuity**: Smooth evolution between timesteps
- **Rate of Change**: Topography shouldn't change too rapidly
- **Geological Plausibility**: Compare with known events

### Reference Frame Checks:
- **Rotation Magnitude**: Difference between frames should be modest
- **Plate Boundaries**: Should remain consistent
- **Coastline Matching**: Continental outlines should align

---
## 📁 Output File Naming Convention

```
{ml_model_name}_Model_{MODEL_NAME}_{reconstruction_time}.nc
```

Examples:
- `DL_Model_Merdith1Ga_120.nc` - Deep learning, 120 Ma
- `RF_Model_Merdith1Ga_65.nc` - Random forest, 65 Ma

**Benefits**:
- Easy sorting by time
- Model type clearly identified
- Compatible with GPlates naming expectations

---

## Visualization in GPlates

### Loading NetCDF Grids:

1. **Open GPlates**
2. **File → Import → Import Raster**
3. **Select NetCDF file(s)**
4. **Set reconstruction time** in filename or metadata
5. **Apply color palette** (e.g., ETOPO1)

### Animation:

1. Load multiple timesteps as **Time-Dependent Raster**
2. Use **Animation Controls** to step through time
3. Export frames for video rendering

---


## ⚠️ Common Issues and Solutions

### Issue 1: Out of Memory
**Problem**: Large grids consume too much RAM  
**Solution**: 
- Process in smaller time windows
- Reduce spatial resolution temporarily
- Use chunking in xarray

### Issue 2: Interpolation Artifacts
**Problem**: Unrealistic values near data boundaries  
**Solution**:
- Reduce `threshold_distance` in post_process_grid()
- Increase smoothing radius carefully
- Check for outliers in input data

### Issue 3: Temporal Discontinuities
**Problem**: Sudden jumps between timesteps  
**Solution**:
- Increase temporal smoothing window
- Check for missing data files
- Verify model predictions are reasonable

### Issue 4: Reference Frame Mismatch
**Problem**: Grids don't align with plate boundaries  
**Solution**:
- Double-check anchor plate IDs
- Verify rotation files are correct
- Use diagnostic plots to compare frames

---

## 🔬 Advanced Options

### Variable Smoothing Parameters:
Different regions may need different smoothing:
```python
# Less smoothing in tectonically active regions
if np.abs(elevation) > 2000:  # Mountainous
    sigma = 3
else:  # Lowlands
    sigma = 6
```

### Adaptive Grid Resolution:
Higher resolution near margins:
```python
if trench_distance < 200000:  # Within 200 km of trench
    grid_resolution = 0.1
else:
    grid_resolution = 0.5
```

### Multi-Model Ensembles:
Average predictions from multiple models:
```python
ensemble_mean = (DL_elevation + RF_elevation) / 2
ensemble_std = np.std([DL_elevation, RF_elevation], axis=0)
```

---

## 📈 Performance Tips

### Memory Optimization:
- Process timesteps sequentially for long series
- Use `xarray.open_dataset()` with `chunks` parameter
- Close datasets explicitly: `ds.close()`

### Speed Optimization:
- Use parallel processing for independent timesteps
- Pre-compute interpolation weights
- Store intermediate results (checkpoints)


### Storage Optimization:
- Always use zlib compression
- Consider reducing temporal resolution (5 Ma instead of 1 Ma)
- Archive old runs on external storage

---

## Workflow Order

1. **Setup**: Load configuration and plate models
2. **Spatial Processing**: Grid and smooth each timestep
3. **Temporal Processing**: Interpolate and smooth time series
4. **Reference Conversion**: Rotate to desired frame (optional)
5. **Quality Control**: Visual inspection and validation
6. **Export**: Save final grids with documentation

**Estimated Runtime**:
- Spatial gridding: 2-5 min per timestep
- Temporal smoothing: 30-60 min per 100 Ma window
- Reference conversion: 1-3 min per timestep

---


## Data Flow Diagram

```
Workflow 2 Predictions (Parquet)
         ↓
   Spatial Gridding
         ↓
  Local Interpolation
         ↓
  Gaussian Smoothing
         ↓
  NetCDF (Mantle Frame)
         ↓
  Temporal Interpolation
         ↓
  Temporal Smoothing
         ↓
  NetCDF (Smoothed)
         ↓
  Reference Conversion
         ↓
  NetCDF (Paleomag Frame)
         ↓
   GPlates Visualization
```

---
